# Coverage BBox + Datetime Query

**Persona: GS — Geospatial Scientist**

## Goal

1. Inspect a collection's **domain set** (spatial/temporal extent and grid structure).
2. Retrieve a **bbox + datetime slice** of the coverage and validate the CoverageJSON response.
3. Confirm correct handling of out-of-extent requests and point-in-time queries.

## Prerequisites

- A running GeoID / DynaStore instance with the **CoveragesService** mounted.
- The target collection's driver must advertise at least the `INTROSPECTION` capability for range-type queries.
  (A driver without `INTROSPECTION` will return **501** from `/coverage/rangetype` — handled gracefully below.)
- Data must exist within the queried bbox; otherwise the coverage slice returns an empty but valid response.
- `.env` file with:
  - `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`
- `pip install httpx python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv
import httpx

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN         = os.environ.get("DYNASTORE_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {TOKEN}"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

# Convenience prefix for all coverage routes
COV_PREFIX = f"/coverages/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"

print(f"Coverage prefix: {BASE_URL}{COV_PREFIX}")

## Step 1 — Inspect Domain Set

The **domain set** describes the coverage's coordinate reference system, spatial extent, and — for gridded products — the grid geometry.  
For dekadal rainfall and NDVI mosaics this typically exposes a regular lat/lon grid aligned to 0.05° or 0.1° cells.

In [ ]:
resp_ds = client.get(f"{COV_PREFIX}/coverage/domainset")
assert resp_ds.status_code == 200, (
    f"domainset request failed: {resp_ds.status_code}: {resp_ds.text}"
)

ds = resp_ds.json()
domain_type = ds.get("type", ds.get("domainType", "unknown"))
print(f"Domain type : {domain_type}")

# Print spatial envelope if present
envelope = ds.get("generalGrid", {}).get("envelope", ds.get("envelope", {}))
if envelope:
    low  = envelope.get("low",  envelope.get("lowerBound",  "?"))
    high = envelope.get("high", envelope.get("upperBound", "?"))
    srs  = envelope.get("srsName", envelope.get("crs", "?"))
    print(f"Extent      : {low} → {high}  (CRS: {srs})")
else:
    print("Envelope    : not present in response")

# Print temporal axis if present
axes = ds.get("generalGrid", {}).get("axis", [])
for ax in axes:
    if ax.get("axisLabel", "").lower() in ("time", "t", "date"):
        print(f"Time axis   : {ax}")

In [ ]:
# --- Range type (requires INTROSPECTION capability) ---
resp_rt = client.get(f"{COV_PREFIX}/coverage/rangetype")

if resp_rt.status_code == 200:
    rt = resp_rt.json()
    fields = rt.get("field", [])
    print(f"Range type fields ({len(fields)}):")
    for f in fields:
        uom = f.get("uom", {}).get("code", "-")
        print(f"  {f.get('name', '?')}  ({uom})  — {f.get('definition', '')}")
elif resp_rt.status_code == 501:
    print("rangetype: HTTP 501 — driver does not implement INTROSPECTION.")
    print("Fallback: query /stac/catalogs/.../collections/{id}/items and inspect 'properties' for band metadata.")
else:
    raise AssertionError(
        f"Unexpected status for rangetype: {resp_rt.status_code}: {resp_rt.text}"
    )

## Step 2 — Query Coverage Slice

Request a **spatial + temporal subset** of the coverage.  
GeoID's CoveragesService translates the OGC API – Coverages `bbox` and `datetime` parameters into driver-specific range queries.  
The response `Content-Type` should be `application/json` with a `CoverageJSON` body, or a domain-specific media type (e.g. `image/tiff`).

In [ ]:
# Horn of Africa sub-region: Ethiopia highlands
QUERY_BBOX     = "37.0,8.0,43.0,15.0"   # minLon,minLat,maxLon,maxLat
QUERY_DATETIME = "2024-02-01T00:00:00Z/2024-02-28T23:59:59Z"

resp_cov = client.get(
    f"{COV_PREFIX}/coverage",
    params={"bbox": QUERY_BBOX, "datetime": QUERY_DATETIME}
)
assert resp_cov.status_code == 200, (
    f"Coverage slice failed: {resp_cov.status_code}: {resp_cov.text[:500]}"
)

content_type = resp_cov.headers.get("content-type", "")
assert "coverage" in content_type.lower() or "json" in content_type.lower() or "tiff" in content_type.lower(), (
    f"Unexpected Content-Type: {content_type!r}. Expected coverage/JSON/TIFF media type."
)

print(f"Content-Type : {content_type}")
print(f"Response size: {len(resp_cov.content)} bytes")

In [ ]:
# --- Validate CoverageJSON structure (if JSON response) ---
if "json" in content_type.lower():
    cov_json = resp_cov.json()
    # OGC CoverageJSON must have a 'type' key
    assert "type" in cov_json, (
        f"CoverageJSON response missing required 'type' key. Keys present: {list(cov_json.keys())}"
    )
    cov_type = cov_json["type"]
    print(f"Coverage type : {cov_type}")

    # domainset or domain must be present
    assert "domainset" in cov_json or "domain" in cov_json, (
        "CoverageJSON must include 'domainset' or 'domain'. "
        f"Keys found: {list(cov_json.keys())}"
    )
    print("domainset/domain key present.")

    # rangeset or range values
    assert "rangeset" in cov_json or "ranges" in cov_json, (
        "CoverageJSON must include 'rangeset' or 'ranges'. "
        f"Keys found: {list(cov_json.keys())}"
    )
    print("rangeset/ranges key present.")
else:
    print(f"Binary response ({content_type}) — skipping JSON structure validation.")
    print("Use GDAL or rasterio to inspect binary coverage outputs.")

## Edge Cases

The OGC API – Coverages specification requires that out-of-extent requests return an **empty but valid** coverage response rather than a 4xx error.  
A point-in-time `datetime` (no slash) must also be accepted as a valid temporal filter.

In [ ]:
# --- bbox entirely outside collection extent → empty response, not 4xx ---
# Far North Atlantic — no FAO crop data expected here
OUT_OF_EXTENT_BBOX = "-30.0,60.0,-20.0,70.0"

resp_oe = client.get(
    f"{COV_PREFIX}/coverage",
    params={"bbox": OUT_OF_EXTENT_BBOX, "datetime": QUERY_DATETIME}
)
# Must not return a client error
assert resp_oe.status_code < 400, (
    f"Out-of-extent bbox must not produce 4xx. Got {resp_oe.status_code}: {resp_oe.text[:300]}"
)
print(f"Out-of-extent bbox → HTTP {resp_oe.status_code} (expected 200 with empty/null coverage)")

if "json" in resp_oe.headers.get("content-type", "").lower():
    oe_json = resp_oe.json()
    print(f"Response keys: {list(oe_json.keys()) if isinstance(oe_json, dict) else 'list/other'}")

In [ ]:
# --- Point-in-time datetime (no slash) ---
POINT_DATETIME = "2024-01-11T00:00:00Z"   # Dekad 1 of January 2024

resp_pt = client.get(
    f"{COV_PREFIX}/coverage",
    params={"bbox": QUERY_BBOX, "datetime": POINT_DATETIME}
)
assert resp_pt.status_code == 200, (
    f"Point datetime query failed: {resp_pt.status_code}: {resp_pt.text[:300]}"
)
print(f"Point datetime '{POINT_DATETIME}' → HTTP {resp_pt.status_code}")
print(f"Response size: {len(resp_pt.content)} bytes")